# IBM Nighthawk ZNE+twirl mitigated generator (qisk)

Original qisk script `vqe_zne_mitiq_v5.py`. Runs the closed-loop noisy VQE on FakeNighthawk with zero-noise extrapolation (gate-folding) + Pauli twirling via Mitiq, and writes `vqe_zne_mitiq_v5_raw.pkl`, stored here as `results/hardware/ibm_nighthawk_zne_raw.pkl` (Fig 10). Requires qiskit + aer + ibm-runtime + mitiq (NOT needed to render figures).

In [ ]:
"""
VQE scaling analysis: native Pauli-exponential (HVA) ansatz + pluggable noise mitigation.

Ansatz: RXXGate / RYYGate / RZZGate — transpiler applies KAK decomposition.
6 params/layer (3 shared odd bonds, 3 shared even bonds) — correct HVA symmetry.

Mitigation flags:
  USE_ZNE            — Zero-Noise Extrapolation via Mitiq factories
  USE_PAULI_TWIRLING — Pauli twirling on native 2Q gate (cx/cz/ecr)

Outputs:
  vqe_zne_mitiq_v5_raw.pkl
  vqe_zne_mitiq_v5_summary.json
  Figs/v5_convergence.pdf
  Figs/v5_Einf_vs_Ng.pdf
  Figs/v5_mu_vs_Ng.pdf
"""

import warnings
warnings.filterwarnings("ignore", message=".*no effect in local testing mode.*")
warnings.filterwarnings("ignore", message=".*Properties of fake.*")

import os
import pickle
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from itertools import combinations
from scipy.optimize import minimize, curve_fit
from joblib import Parallel, delayed

from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import RXXGate, RYYGate, RZZGate
from qiskit.transpiler import generate_preset_pass_manager, Layout, PassManager
from qiskit.transpiler.passes import BasisTranslator
from qiskit.circuit.equivalence_library import SessionEquivalenceLibrary as _SEL

from qiskit_ibm_runtime.fake_provider import (
    FakeNighthawk, FakeKyiv, FakeTorino, FakeSherbrooke
)
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit.primitives import StatevectorEstimator
from qiskit_ibm_runtime import EstimatorV2 as Estimator

from mitiq.zne.inference import RichardsonFactory, LinearFactory, ExpFactory

os.makedirs("Figs", exist_ok=True)
plt.rcParams.update({"font.size": 18})
plt.rcParams["text.usetex"] = False


# ── Mitigation flags ──────────────────────────────────────────────────────────

USE_ZNE            = True    # Zero-Noise Extrapolation via Mitiq factories
USE_PAULI_TWIRLING = True   # Pauli twirling on CX gates

# ZNE sub-options (only relevant when USE_ZNE = True)
ZNE_SCALE_FACTORS  = [1, 3]       # odd integers; add 5 for Richardson/cubic
ZNE_FACTORY_TYPE   = "linear"     # "linear" | "richardson" | "exponential"
ZNE_FOLDING_METHOD = "gate"     # "global" | "gate"

# Twirling sub-options (only relevant when USE_PAULI_TWIRLING = True)
N_TWIRL_CIRCUITS   = 8

# ── Experiment config ─────────────────────────────────────────────────────────

LAYER_DEPTHS = [1, 2, 3, 4, 5, 6]
NUM_SEEDS    = 20


# ── Hamiltonian ───────────────────────────────────────────────────────────────

def heisenberg_hamiltonian(nqbt):
    paulis, coeffs = [], []
    for i in range(nqbt - 1):
        paulis += [
            "I"*i + "XX" + "I"*(nqbt-i-2),
            "I"*i + "YY" + "I"*(nqbt-i-2),
            "I"*i + "ZZ" + "I"*(nqbt-i-2),
        ]
        coeffs += [1.0, 1.0, 1.0]
    return SparsePauliOp(paulis, coeffs)


# ── Variational circuit (HVA, native Pauli-exponential form) ──────────────────
#
# Each HVA layer applies e^{iθ XX}, e^{iθ YY}, e^{iθ ZZ} on each bond using
# RXXGate / RYYGate / RZZGate. The transpiler sees the 2Q unitary directly
# and decomposes it via KAK, typically using 1 CX (for a single Pauli exp)
# rather than 2 CX (manual CX-Rz-CX sandwich). Consecutive same-axis gates on
# the same qubit pair are sometimes further merged.
#
# Parameter count: 3 params per bond per layer × (nqbt-1) bonds × ct layers.
# Odd bonds (1-2, 3-4, ...) and even bonds (0-1, 2-3, ...) are applied in
# separate sub-layers to mirror the original HVA structure.

def variational_circuit(nqbt, ct):
    # 6 params per layer: 3 shared across all odd bonds, 3 shared across all even bonds.
    # This is the correct HVA structure — translational symmetry of the Heisenberg chain
    # means all bonds in the same parity sublayer are governed by the same angles.
    n_params = 6 * ct
    params   = ParameterVector("θ", n_params)
    qc       = QuantumCircuit(nqbt)

    for q in range(nqbt):
        qc.x(q)
    for q in range(0, nqbt, 2):
        qc.h(q)
        if q + 1 < nqbt:
            qc.cx(q, q + 1)

    for it in range(ct):
        p = 6 * it
        # Odd bonds: (1,2), (3,4), ... — all share params p, p+1, p+2
        for q in range(1, nqbt - 1, 2):
            qc.append(RXXGate(params[p + 0]), [q, q + 1])
            qc.append(RYYGate(params[p + 1]), [q, q + 1])
            qc.append(RZZGate(params[p + 2]), [q, q + 1])
        # Even bonds: (0,1), (2,3), ... — all share params p+3, p+4, p+5
        for q in range(0, nqbt - 1, 2):
            qc.append(RXXGate(params[p + 3]), [q, q + 1])
            qc.append(RYYGate(params[p + 4]), [q, q + 1])
            qc.append(RZZGate(params[p + 5]), [q, q + 1])

    return qc


# ── Gate counting (manuscript convention) ────────────────────────────────────

def count_gates(circuit: QuantumCircuit) -> int:
    """Ng = Σ_gates (1 for 1Q, 2 for 2Q). Matches manuscript convention."""
    ng = 0
    for inst in circuit.data:
        ng += 2 if inst.operation.num_qubits >= 2 else 1
    return ng


# ── Backend selection: pick QPU with lowest mean native 2Q error ──────────────

num_qubits = 5

_CANDIDATE_BACKENDS = [FakeNighthawk, FakeKyiv, FakeTorino, FakeSherbrooke]

def _native_2q_gate(backend_instance):
    """Return the primary native 2Q gate name for this backend."""
    gates_2q = {
        g["gate"]
        for g in backend_instance.properties().to_dict().get("gates", [])
        if len(g["qubits"]) == 2
    }
    for preferred in ("cx", "ecr", "cz", "rzz"):
        if preferred in gates_2q:
            return preferred
    return next(iter(gates_2q), "cx")

def _mean_2q_error(backend_instance):
    native = _native_2q_gate(backend_instance)
    errors = [
        g["parameters"][0]["value"]
        for g in backend_instance.properties().to_dict().get("gates", [])
        if g["gate"] == native
    ]
    return sum(errors) / len(errors) if errors else float("inf")

def select_best_qubits(backend_instance, nq):
    native = _native_2q_gate(backend_instance)
    gate_errors = {
        tuple(gate["qubits"]): gate["parameters"][0]["value"]
        for gate in backend_instance.properties().to_dict().get("gates", [])
        if gate["gate"] == native
    }
    best, best_err = None, float("inf")
    for qs in combinations(range(backend_instance.num_qubits), nq):
        err = sum(gate_errors.get((qs[i], qs[i+1]), 1.0) for i in range(nq - 1))
        if err < best_err:
            best_err, best = err, qs
    return list(best)

print("=" * 60)
print("QPU selection by mean native 2Q error rate")
print("=" * 60)
_backend_scores = []
for cls in _CANDIDATE_BACKENDS:
    b = cls()
    native   = _native_2q_gate(b)
    mean_err = _mean_2q_error(b)
    best_qs  = select_best_qubits(b, num_qubits)
    gate_errs = {
        tuple(g["qubits"]): g["parameters"][0]["value"]
        for g in b.properties().to_dict().get("gates", [])
        if g["gate"] == native
    }
    best_chain_err = sum(
        gate_errs.get((best_qs[i], best_qs[i+1]), 1.0) for i in range(num_qubits - 1)
    )
    _backend_scores.append((mean_err, cls, b, best_qs, best_chain_err))
    print(f"  {b.name:<22}  native={native:<3}  mean 2Q = {mean_err:.5f}  "
          f"best-chain = {best_chain_err:.5f}  qubits = {best_qs}")

_backend_scores.sort(key=lambda x: x[0])
_, _best_cls, backend, selected_qubits, _ = _backend_scores[0]
print(f"\n  Selected backend        : {backend.name}  (lowest mean 2Q error)")
print("=" * 60)

noise_model = NoiseModel.from_backend(backend)
print(f"  Basis gates with errors : {noise_model.basis_gates}")
print(f"  Qubits with errors      : {sorted(noise_model.noise_qubits)[:10]} ...")
print(f"  Selected qubits         : {selected_qubits}")
print("=" * 60)

noisy_sim = AerSimulator(method="statevector").from_backend(backend)

qr = QuantumRegister(num_qubits, "q")
initial_layout = Layout({qr[i]: q for i, q in enumerate(selected_qubits)})
pm = generate_preset_pass_manager(
    backend=noisy_sim,
    optimization_level=3,
    layout_method="trivial",
    initial_layout=initial_layout,
    qubits_initially_zero=True,
)

# Translates non-basis gates produced by inversion during folding (e.g. sxdg)
# back to the backend basis. No re-routing — qubit layout is untouched.
_BASIS_GATES = list(set(noise_model.basis_gates) | {"id", "barrier"})
_basis_pm    = PassManager([BasisTranslator(_SEL, _BASIS_GATES)])


# ── Exact ground state energy ─────────────────────────────────────────────────

obs_exact    = heisenberg_hamiltonian(num_qubits)
ham_matrix   = np.array(obs_exact)
exact_energy = min(np.linalg.eig(ham_matrix)[0]).real
print(f"Exact ground state energy: E_gs = {exact_energy:.6f}")

# ── Derived labels ────────────────────────────────────────────────────────────

_mit_parts = []
if USE_ZNE:
    _mit_parts.append(f"ZNE/{ZNE_FACTORY_TYPE}/{ZNE_FOLDING_METHOD}-fold")
if USE_PAULI_TWIRLING:
    _mit_parts.append(f"Twirl(n={N_TWIRL_CIRCUITS})")
MITIGATION_LABEL = "+".join(_mit_parts) if _mit_parts else "noisy (no mitigation)"

print(f"Active mitigation        : {MITIGATION_LABEL}")
print(f"ZNE scale factors        : {ZNE_SCALE_FACTORS}\n")


# ── Circuit folding ───────────────────────────────────────────────────────────

def fold_circuit_global(circuit: QuantumCircuit, scale_factor: int) -> QuantumCircuit:
    """Global folding: C → C (C†C)^k, k=(λ-1)/2. No re-routing."""
    assert scale_factor >= 1 and scale_factor % 2 == 1, "scale_factor must be odd ≥ 1"
    k = (scale_factor - 1) // 2
    if k == 0:
        return circuit.copy()
    inv = circuit.inverse()
    folded = circuit.copy()
    for _ in range(k):
        folded = folded.compose(inv).compose(circuit)
    return folded


def fold_circuit_gate(circuit: QuantumCircuit, scale_factor: int) -> QuantumCircuit:
    """Gate-level folding: G → G (G†G)^k per gate. No re-routing."""
    assert scale_factor >= 1 and scale_factor % 2 == 1, "scale_factor must be odd ≥ 1"
    k = (scale_factor - 1) // 2
    if k == 0:
        return circuit.copy()
    folded = QuantumCircuit(*circuit.qregs, *circuit.cregs)
    for inst in circuit.data:
        folded.append(inst)
        if inst.operation.name in ("barrier", "measure", "reset"):
            continue
        try:
            inv_op = inst.operation.inverse()
        except Exception:
            continue
        for _ in range(k):
            folded.append(inv_op, inst.qubits, inst.clbits)
            folded.append(inst.operation.copy(), inst.qubits, inst.clbits)
    return folded


def fold_circuit(circuit: QuantumCircuit, scale_factor: int) -> QuantumCircuit:
    """
    Fold and translate any non-basis gates back to the backend basis.
    No re-routing — coupling map and qubit layout are preserved.
    """
    if ZNE_FOLDING_METHOD == "gate":
        folded = fold_circuit_gate(circuit, scale_factor)
    else:
        folded = fold_circuit_global(circuit, scale_factor)
    return _basis_pm.run(folded)


# ── Pauli twirling ────────────────────────────────────────────────────────────
#
# Supports CX, CZ, and ECR — the three native 2Q gates found on the candidate
# backends. Each gate is a Clifford, so Pauli twirling is exact.
#
# Encoding: 0=I, 1=X, 2=Y, 3=Z  in (x_bit, z_bit) representation.
# CX conjugation: ctrl→(cx, cz^tz), tgt→(tx^cx, tz)
# CZ conjugation: ctrl→(cx, cz^tx), tgt→(tx, tz^cx)   [CZ is symmetric]
# ECR conjugation: lookup table (computed from Heisenberg propagation).

_P_ENC  = {0: (0, 0), 1: (1, 0), 2: (1, 1), 3: (0, 1)}
_P_DEC  = {v: k for k, v in _P_ENC.items()}
_P_GATE = {0: "id", 1: "x", 2: "y", 3: "z"}


def _cx_conjugate(p_c: int, p_t: int) -> tuple[int, int]:
    c_x, c_z = _P_ENC[p_c]
    t_x, t_z = _P_ENC[p_t]
    return _P_DEC[(c_x, c_z ^ t_z)], _P_DEC[(t_x ^ c_x, t_z)]


def _cz_conjugate(p_c: int, p_t: int) -> tuple[int, int]:
    c_x, c_z = _P_ENC[p_c]
    t_x, t_z = _P_ENC[p_t]
    return _P_DEC[(c_x, c_z ^ t_x)], _P_DEC[(t_x, t_z ^ c_x)]


# ECR Heisenberg propagation table: (p_ctrl, p_tgt) -> (p_ctrl_out, p_tgt_out)
# Verified numerically via gate @ (P_c x P_t) @ gate†
_ECR_CONJ = {
    (0,0):(0,0), (0,1):(1,2), (0,2):(1,1), (0,3):(0,3),
    (1,0):(1,0), (1,1):(0,2), (1,2):(0,1), (1,3):(1,3),
    (2,0):(3,3), (2,1):(2,1), (2,2):(2,2), (2,3):(3,0),
    (3,0):(2,3), (3,1):(3,1), (3,2):(3,2), (3,3):(2,0),
}


def twirl_2q_gate(circuit: QuantumCircuit, rng: np.random.Generator) -> QuantumCircuit:
    """
    Pauli-twirl the native 2Q gate (cx / cz / ecr). Net unitary unchanged;
    coherent 2Q noise is converted to Pauli noise by averaging over twirl circuits.
    """
    twirled = QuantumCircuit(*circuit.qregs, *circuit.cregs)
    for inst in circuit.data:
        name = inst.operation.name
        if name in ("cx", "cz", "ecr"):
            q0, q1 = inst.qubits
            p_c = int(rng.integers(4))
            p_t = int(rng.integers(4))
            if name == "cx":
                q_c, q_t = _cx_conjugate(p_c, p_t)
            elif name == "cz":
                q_c, q_t = _cz_conjugate(p_c, p_t)
            else:  # ecr
                q_c, q_t = _ECR_CONJ[(p_c, p_t)]
            if p_c != 0:
                getattr(twirled, _P_GATE[p_c])(q0)
            if p_t != 0:
                getattr(twirled, _P_GATE[p_t])(q1)
            twirled.append(inst)
            if q_c != 0:
                getattr(twirled, _P_GATE[q_c])(q0)
            if q_t != 0:
                getattr(twirled, _P_GATE[q_t])(q1)
        else:
            twirled.append(inst)
    return _basis_pm.run(twirled)


# ── ZNE factory ───────────────────────────────────────────────────────────────

def build_factory(factory_type: str, scale_factors: list):
    if factory_type == "richardson":
        return RichardsonFactory(scale_factors)
    elif factory_type == "linear":
        return LinearFactory(scale_factors)
    elif factory_type == "exponential":
        return ExpFactory(scale_factors, asymptote=exact_energy)
    else:
        raise ValueError(f"Unknown factory_type: {factory_type!r}")


# ── Energy estimators ─────────────────────────────────────────────────────────

def _run_single(circuit, observable, estimator) -> float:
    return float(np.squeeze(
        estimator.run([(circuit, observable)]).result()[0].data.evs
    ))


def _run_twirled_average(circuit, observable, estimator, n: int) -> float:
    rng_base = np.random.default_rng()
    pubs = [
        (twirl_2q_gate(circuit, np.random.default_rng(rng_base.integers(2**31))), observable)
        for _ in range(n)
    ]
    results = estimator.run(pubs).result()
    return float(np.mean([np.squeeze(results[i].data.evs) for i in range(n)]))


def zne_energy(bound_circuit, observable, estimator) -> float:
    """
    ZNE via Mitiq factory. Folds without re-routing; optionally twirls each
    folded circuit. Extrapolates to λ=0.
    """
    factory = build_factory(ZNE_FACTORY_TYPE, ZNE_SCALE_FACTORS)
    if USE_PAULI_TWIRLING:
        for c in ZNE_SCALE_FACTORS:
            folded = fold_circuit(bound_circuit, int(c))
            energy = _run_twirled_average(folded, observable, estimator, N_TWIRL_CIRCUITS)
            factory.push({"scale_factor": float(c)}, energy)
    else:
        pubs = [(fold_circuit(bound_circuit, int(c)), observable) for c in ZNE_SCALE_FACTORS]
        results = estimator.run(pubs).result()
        for i, c in enumerate(ZNE_SCALE_FACTORS):
            factory.push({"scale_factor": float(c)},
                         float(np.squeeze(results[i].data.evs)))
    return float(factory.reduce())


# ── Cost functions ────────────────────────────────────────────────────────────

def cost_func_plain(params, ansatz, hamiltonian, estimator, cost_history):
    energy = _run_single(ansatz.assign_parameters(params), hamiltonian, estimator)
    cost_history.append(energy)
    return energy


def cost_func_twirl_only(params, ansatz, hamiltonian, estimator, cost_history):
    energy = _run_twirled_average(
        ansatz.assign_parameters(params), hamiltonian, estimator, N_TWIRL_CIRCUITS
    )
    cost_history.append(energy)
    return energy


def cost_func_zne(params, ansatz, hamiltonian, estimator, cost_history):
    energy = zne_energy(ansatz.assign_parameters(params), hamiltonian, estimator)
    cost_history.append(energy)
    return energy


def _select_cost_fn():
    if USE_ZNE:
        return cost_func_zne
    if USE_PAULI_TWIRLING:
        return cost_func_twirl_only
    return cost_func_plain

ACTIVE_COST_FN = _select_cost_fn()


# ── Seed runners ──────────────────────────────────────────────────────────────

def _minimize(cost_fn, params, ansatz, observable, estimator, cost_history):
    bounds = [(0, 2 * np.pi)] * len(params)
    res = minimize(
        cost_fn, x0=params,
        args=(ansatz, observable, estimator, cost_history),
        method="cobyla", bounds=bounds,
        options={"maxiter": 10000, "tol": 1e-8, "disp": False},
    )
    return cost_history, res.fun


def run_seed_noiseless(seed_idx, ansatz_logical, observable_logical):
    est = StatevectorEstimator()
    np.random.seed(seed_idx)
    params = 2 * np.pi * np.random.random(ansatz_logical.num_parameters)
    return _minimize(cost_func_plain, params, ansatz_logical, observable_logical, est, [])


def run_seed_mitigated(seed_idx, ansatz_ibm, observable_ibm):
    _backend = _best_cls()
    sim = AerSimulator(method="statevector").from_backend(_backend)
    est = Estimator(sim)
    np.random.seed(seed_idx)
    params = 2 * np.pi * np.random.random(ansatz_ibm.num_parameters)
    return _minimize(ACTIVE_COST_FN, params, ansatz_ibm, observable_ibm, est, [])


# ── Main loop ─────────────────────────────────────────────────────────────────

noiseless_results = {}
mitigated_results = {}
Ng_by_layer       = {}

for layer_depth in LAYER_DEPTHS:
    ansatz         = variational_circuit(num_qubits, layer_depth)
    ansatz_ibm     = pm.run(ansatz)
    observable_ibm = heisenberg_hamiltonian(num_qubits).apply_layout(ansatz_ibm.layout)
    Ng             = count_gates(ansatz_ibm)
    Ng_by_layer[layer_depth] = Ng

    print(f"\n{'='*60}")
    print(f"Layer depth {layer_depth}  |  Ng = {Ng} gates  "
          f"(params = {ansatz_ibm.num_parameters})")
    print("="*60)

    print("  [noiseless]")
    observable_logical = heisenberg_hamiltonian(num_qubits)
    nl_res = Parallel(n_jobs=-1)(
        delayed(run_seed_noiseless)(s, ansatz, observable_logical)
        for s in range(NUM_SEEDS)
    )
    nl_hists, nl_finals = zip(*nl_res)
    noiseless_results[layer_depth] = {
        "cost_histories": nl_hists, "final_energies": nl_finals, "Ng": Ng
    }
    print(f"  Noiseless avg  : {np.mean(nl_finals):.4f} ± {np.std(nl_finals):.4f}")

    print(f"  [{MITIGATION_LABEL}]")
    mit_res = Parallel(n_jobs=-1)(
        delayed(run_seed_mitigated)(s, ansatz_ibm, observable_ibm)
        for s in range(NUM_SEEDS)
    )
    mit_hists, mit_finals = zip(*mit_res)
    mitigated_results[layer_depth] = {
        "cost_histories": mit_hists, "final_energies": mit_finals, "Ng": Ng
    }
    print(f"  Mitigated avg  : {np.mean(mit_finals):.4f} ± {np.std(mit_finals):.4f}")


# ── Step 3: Fit noiseless scaling ─────────────────────────────────────────────

layer_list = sorted(noiseless_results.keys())
Ng_arr     = np.array([noiseless_results[nl]["Ng"] for nl in layer_list], dtype=float)
E_inf_nl   = np.array([np.mean(noiseless_results[nl]["final_energies"]) for nl in layer_list])

def E_inf_nl_model(Ng, beta, kappa):
    return beta * np.exp(-kappa * Ng) + exact_energy

try:
    popt_inf, _ = curve_fit(
        E_inf_nl_model, Ng_arr, E_inf_nl,
        p0=[4.0, 0.01], bounds=([0, 1e-6], [50, 1.0]), maxfev=10000,
    )
    beta_fit, kappa_fit = popt_inf
    print(f"\n[Noiseless fit] beta = {beta_fit:.4f}, kappa = {kappa_fit:.5f}")
except RuntimeError as e:
    print(f"[WARNING] E_inf noiseless fit failed: {e}. Using fallback β=3.94, κ=0.01")
    beta_fit, kappa_fit = 3.94, 0.01


# ── Step 3b: Convergence rate μ per layer ────────────────────────────────────

def fit_mu_from_history(cost_history, E_inf, max_iters=None):
    t  = np.arange(len(cost_history), dtype=float)
    dE = np.array(cost_history, dtype=float) - E_inf
    if max_iters is not None:
        t, dE = t[:max_iters], dE[:max_iters]
    valid = dE > 1e-8
    if valid.sum() < 5:
        return np.nan
    try:
        popt, _ = curve_fit(
            lambda t, alpha, mu: alpha * np.exp(-mu * t),
            t[valid], dE[valid],
            p0=[max(dE[valid]), 0.05],
            bounds=([0, 1e-8], [np.inf, 10.0]),
            maxfev=10000,
        )
        return popt[1]
    except RuntimeError:
        return np.nan


def fit_mu_power_law(Ng_arr, mu_arr):
    best = {"r2": -np.inf, "mu0": np.nan, "lam": np.nan, "ng_th": 0}
    valid_mu = np.isfinite(mu_arr) & (mu_arr > 0)
    for ng_th in np.arange(0, int(Ng_arr.min() * 0.8), 10):
        x = Ng_arr - ng_th
        if np.any(x[valid_mu] <= 0):
            continue
        log_x  = np.log(x[valid_mu])
        log_mu = np.log(mu_arr[valid_mu])
        if len(log_x) < 3:
            continue
        slope, intercept = np.polyfit(log_x, log_mu, 1)
        pred   = slope * log_x + intercept
        ss_res = np.sum((log_mu - pred)**2)
        ss_tot = np.sum((log_mu - log_mu.mean())**2)
        r2     = 1.0 - ss_res / ss_tot if ss_tot > 1e-12 else 0.0
        if r2 > best["r2"]:
            best = {"r2": r2, "mu0": np.exp(intercept), "lam": -slope, "ng_th": ng_th}
    return best["mu0"], best["lam"], best["ng_th"], best["r2"]


print("\n[Convergence rate μ]")
mu_nl_per_layer  = []
mu_mit_per_layer = []

for nl in layer_list:
    Ng, max_iter = noiseless_results[nl]["Ng"], 25 * nl
    E_inf     = np.mean(noiseless_results[nl]["final_energies"])
    E_inf_mit = np.mean(mitigated_results[nl]["final_energies"])

    mu_nl  = np.nanmean([fit_mu_from_history(h, E_inf,     max_iters=max_iter)
                         for h in noiseless_results[nl]["cost_histories"]])
    mu_mit = np.nanmean([fit_mu_from_history(h, E_inf_mit, max_iters=max_iter)
                         for h in mitigated_results[nl]["cost_histories"]])
    mu_nl_per_layer.append(mu_nl)
    mu_mit_per_layer.append(mu_mit)
    print(f"  Layer {nl} (Ng={Ng}): mu_nl={mu_nl:.4f}  mu_mit={mu_mit:.4f}")

mu_nl_arr  = np.array(mu_nl_per_layer)
mu_mit_arr = np.array(mu_mit_per_layer)

mu0_fit, lam_fit, ng_th_fit, r2_fit = fit_mu_power_law(Ng_arr, mu_nl_arr)
print(f"\n[mu power-law fit] mu_0={mu0_fit:.4f}, lambda={lam_fit:.4f}, "
      f"Ng_th={ng_th_fit}, R²={r2_fit:.4f}")


# ── Step 4: Extract eps_eff ───────────────────────────────────────────────────

E_inf_mit_arr       = np.array([np.mean(mitigated_results[nl]["final_energies"]) for nl in layer_list])
E_inf_nl_model_vals = beta_fit * np.exp(-kappa_fit * Ng_arr) + exact_energy

ratios = E_inf_mit_arr / E_inf_nl_model_vals
valid  = (ratios > 0) & np.isfinite(ratios) & (E_inf_nl_model_vals < 0) & (E_inf_mit_arr < 0)

eps_eff = np.nan
N_g_opt = np.nan
if valid.sum() >= 2:
    slope, _  = np.polyfit(Ng_arr[valid], np.log(ratios[valid]), 1)
    eps_eff   = 1.0 - np.exp(slope)
    if eps_eff > 0:
        N_g_opt = (1.0 / kappa_fit) * np.log(
            beta_fit * (kappa_fit + eps_eff) / (abs(exact_energy) * eps_eff)
        )


# ── Summary table ─────────────────────────────────────────────────────────────

W = 66
print("\n" + "=" * W)
print(f"{'FITTED PARAMETERS SUMMARY':^{W}}")
print("=" * W)
print(f"  {'Model':<34}  {'Parameter':<12}  {'Value'}")
print("-" * W)
print(f"  {'E_inf^nl = β·exp(-κ·Ng)+E_gs':<34}  {'β':<12}  {beta_fit:.5f}")
print(f"  {'':<34}  {'κ':<12}  {kappa_fit:.6f}")
print("-" * W)
print(f"  {'μ(Ng) = μ₀·(Ng-Ng_th)^(-λ)':<34}  {'μ₀':<12}  {mu0_fit:.5f}")
print(f"  {'':<34}  {'λ':<12}  {lam_fit:.5f}")
print(f"  {'':<34}  {'Ng_th':<12}  {ng_th_fit}")
print(f"  {'':<34}  {'R²':<12}  {r2_fit:.5f}")
print("-" * W)
eps_str = f"{eps_eff:.4e}" if np.isfinite(eps_eff) else "N/A"
opt_str = (f"{N_g_opt:.0f} gates (~{N_g_opt/150:.1f} layers)"
           if np.isfinite(eps_eff) and eps_eff > 0 else "N/A")
print(f"  {'ε_eff (global depol.)':<34}  {'ε_eff':<12}  {eps_str}")
print(f"  {'':<34}  {'Ng_opt (Eq.A11)':<12}  {opt_str}")
print("=" * W)
print(f"\n  E_gs (exact)  = {exact_energy:.6f}")
print(f"  Ansatz        = HVA (RXX/RYY/RZZ, KAK-compiled)")
print(f"  Mitigation    = {MITIGATION_LABEL}")
print(f"  Ng range      = {int(Ng_arr.min())} – {int(Ng_arr.max())}  "
      f"(layers {layer_list[0]}–{layer_list[-1]})")
print("=" * W)

print(f"\n{'Layer':>6} {'Ng':>6} {'E_inf_nl':>10} {'E_inf_mit':>10} "
      f"{'ΔE_nl':>8} {'ΔE_mit':>8} {'μ_nl':>8} {'μ_mit':>8}")
print("-" * 72)
for i, nl in enumerate(layer_list):
    print(f"{nl:>6} {Ng_arr[i]:>6.0f} {E_inf_nl[i]:>10.4f} {E_inf_mit_arr[i]:>10.4f} "
          f"{E_inf_nl[i]-exact_energy:>8.4f} {E_inf_mit_arr[i]-exact_energy:>8.4f} "
          f"{mu_nl_arr[i]:>8.4f} {mu_mit_arr[i]:>8.4f}")


# ── Save ──────────────────────────────────────────────────────────────────────

with open("vqe_zne_mitiq_v5_raw.pkl", "wb") as f:
    pickle.dump({
        "noiseless_results": noiseless_results,
        "mitigated_results":  mitigated_results,
        "Ng_by_layer":        Ng_by_layer,
        "exact_energy":       exact_energy,
        "beta_fit":           beta_fit,
        "kappa_fit":          kappa_fit,
        "mu_nl_arr":          mu_nl_arr.tolist(),
        "mu_mit_arr":         mu_mit_arr.tolist(),
        "eps_eff":            float(eps_eff) if np.isfinite(eps_eff) else None,
        "N_g_opt":            float(N_g_opt) if np.isfinite(N_g_opt) else None,
        "config": {
            "USE_ZNE":            USE_ZNE,
            "USE_PAULI_TWIRLING": USE_PAULI_TWIRLING,
            "ZNE_SCALE_FACTORS":  ZNE_SCALE_FACTORS,
            "ZNE_FACTORY_TYPE":   ZNE_FACTORY_TYPE,
            "ZNE_FOLDING_METHOD": ZNE_FOLDING_METHOD,
            "LAYER_DEPTHS":       LAYER_DEPTHS,
            "NUM_SEEDS":          NUM_SEEDS,
        },
        "backend":         backend.name,
        "selected_qubits": selected_qubits,
    }, f)
print("Saved: vqe_zne_mitiq_v5_raw.pkl")

summary = {}
for ld in layer_list:
    summary[str(ld)] = {
        "Ng":            noiseless_results[ld]["Ng"],
        "E_inf_nl":      float(np.mean(noiseless_results[ld]["final_energies"])),
        "E_inf_nl_std":  float(np.std(noiseless_results[ld]["final_energies"])),
        "E_inf_mit":     float(np.mean(mitigated_results[ld]["final_energies"])),
        "E_inf_mit_std": float(np.std(mitigated_results[ld]["final_energies"])),
        "mu_nl":         float(mu_nl_arr[layer_list.index(ld)]),
        "mu_mit":        float(mu_mit_arr[layer_list.index(ld)]),
    }
with open("vqe_zne_mitiq_v5_summary.json", "w") as f:
    json.dump({
        "summary":      summary,
        "exact_energy": exact_energy,
        "beta_fit":     beta_fit,
        "kappa_fit":    kappa_fit,
        "eps_eff":      float(eps_eff) if np.isfinite(eps_eff) else None,
        "N_g_opt":      float(N_g_opt) if np.isfinite(N_g_opt) else None,
    }, f, indent=2)
print("Saved: vqe_zne_mitiq_v5_summary.json")


# ── Plots ─────────────────────────────────────────────────────────────────────

n_lay    = len(layer_list)
colors   = plt.cm.tab10(np.linspace(0, 0.9, n_lay))
gate_ctt = Ng_arr.astype(int)
Ng_dense = np.linspace(Ng_arr[0] * 0.9, Ng_arr[-1] * 1.1, 300)
fit_ran  = 25
xlim_max = fit_ran * n_lay
xx       = np.arange(0, 25000, dtype=int)


def _padded_mean_std(histories):
    max_l  = max(len(h) for h in histories)
    padded = np.array([list(h) + [h[-1]] * (max_l - len(h)) for h in histories])
    return np.mean(padded, axis=0), np.std(padded, axis=0)


def exp_model(t, a, b):
    return a * np.exp(-b * t)


val_nl_m, val_nl_s   = [], []
val_mit_m, val_mit_s = [], []
nl_mean_traj,  nl_std_traj  = {}, {}
mit_mean_traj, mit_std_traj = {}, {}

for ld in layer_list:
    m, s = _padded_mean_std(noiseless_results[ld]["cost_histories"])
    nl_mean_traj[ld], nl_std_traj[ld] = m, s
    val_nl_m.append(np.mean(noiseless_results[ld]["final_energies"]))
    val_nl_s.append(np.std(noiseless_results[ld]["final_energies"]))

    m, s = _padded_mean_std(mitigated_results[ld]["cost_histories"])
    mit_mean_traj[ld], mit_std_traj[ld] = m, s
    val_mit_m.append(np.mean(mitigated_results[ld]["final_energies"]))
    val_mit_s.append(np.std(mitigated_results[ld]["final_energies"]))

val_nl_m  = np.array(val_nl_m);  val_nl_s  = np.array(val_nl_s)
val_mit_m = np.array(val_mit_m); val_mit_s = np.array(val_mit_s)

# Plot (a)/(b): convergence trajectories
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)
for ax, (mean_traj, std_traj, val_m, panel, title) in zip(axes, [
    (nl_mean_traj,  nl_std_traj,  val_nl_m,  "a", "Noiseless"),
    (mit_mean_traj, mit_std_traj, val_mit_m, "b", MITIGATION_LABEL),
]):
    for i, (ld, color) in enumerate(zip(layer_list, colors)):
        fit_range = (i + 1) * fit_ran
        yy = mean_traj[ld] - val_m[i]
        sd = std_traj[ld]
        xp, yp, ep = xx[:fit_range], yy[:fit_range], sd[:fit_range]
        ax.errorbar(xp, yp, yerr=ep, fmt="-", color=color,
                    label=r"$%d~(N_\text{g}=%d)$" % (ld, gate_ctt[i]))
        valid = yp > 1e-8
        if valid.sum() >= 5:
            try:
                pfit, _ = curve_fit(
                    exp_model, xp[valid], yp[valid],
                    sigma=ep[valid] + 1e-12, absolute_sigma=True,
                    p0=[max(yp[valid]), 0.05],
                    bounds=([0, 1e-8], [np.inf, 10]), maxfev=10000,
                )
                x_fit = np.linspace(0, fit_range, 200)
                ax.plot(x_fit, exp_model(x_fit, *pfit), "--k", lw=1.2, alpha=0.6)
            except RuntimeError:
                pass
    ax.set_xlabel(r"$\text{Number of Iterations,}~N_\text{it}$", fontsize=18)
    ax.set_ylabel(r"$E(N_\text{it}) - E_{\infty}$", fontsize=18)
    ax.set_xlim(0, xlim_max); ax.set_ylim(bottom=0)
    ax.legend(ncol=2, loc="upper right", title=r"$N_\text{layers}$",
              title_fontsize=16, fontsize=15, labelspacing=0.3, handletextpad=0.3)
    ax.text(0.02, 0.97, f"({panel})", transform=ax.transAxes,
            fontsize=18, verticalalignment="top")
    ax.set_title(title, fontsize=18, pad=8)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("Figs/v5_convergence.pdf", bbox_inches="tight", dpi=300)
plt.show()
print("Saved Figs/v5_convergence.pdf")

# Plot (c): E_inf vs Ng
fig, ax = plt.subplots(figsize=(8, 6))
ax.errorbar(gate_ctt, val_nl_m, yerr=val_nl_s,
            fmt="+-", markersize=10, color="black", linestyle="-",
            label=r"$\epsilon = 0~\text{(noiseless)}$")
ax.errorbar(gate_ctt, val_mit_m, yerr=val_mit_s,
            fmt="o", markersize=10, color="#1E90FF", linestyle="--",
            barsabove=True, label=rf"$\text{{{MITIGATION_LABEL}}}$")
ax.axhline(exact_energy, color="m", linestyle="--", lw=1.5, label=r"$E_\text{gs}$")
ax.set_xlabel(r"$\text{Number of gates, }N_\text{g}$", fontsize=18)
ax.set_ylabel(r"$\text{Converged VQE energy,}~E_\infty$", fontsize=18)
ax.set_xticks(gate_ctt)
ax.set_xticklabels(gate_ctt, rotation=45, ha="right")
axx = ax.secondary_xaxis("top")
axx.set_xticks(gate_ctt)
axx.set_xticklabels(np.arange(1, n_lay + 1, dtype=int))
axx.set_xlabel(r"$N_\text{layers}$", fontsize=18)
ax.legend(ncol=1, loc="lower right", fontsize=16,
          labelspacing=0.4, handletextpad=0.4)
ax.text(0.02, 0.97, "(c)", transform=ax.transAxes,
        fontsize=18, verticalalignment="top")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("Figs/v5_Einf_vs_Ng.pdf", bbox_inches="tight", dpi=300)
plt.show()
print("Saved Figs/v5_Einf_vs_Ng.pdf")

# Plot (d): mu vs Ng
fig, ax = plt.subplots(figsize=(8, 6))
valid_nl  = np.isfinite(mu_nl_arr)
valid_mit = np.isfinite(mu_mit_arr)
ax.scatter(Ng_arr[valid_nl],  mu_nl_arr[valid_nl],  s=80, marker="s",
           color="black",     zorder=5, label=r"$\mu$ noiseless")
ax.scatter(Ng_arr[valid_mit], mu_mit_arr[valid_mit], s=80, marker="o",
           color="#1E90FF",   zorder=5, label=rf"$\mu$ {MITIGATION_LABEL}")
if np.isfinite(mu0_fit) and np.isfinite(lam_fit):
    x_fit = Ng_dense - ng_th_fit
    mask  = x_fit > 0
    ax.plot(Ng_dense[mask], mu0_fit * x_fit[mask]**(-lam_fit), "k--", lw=2,
            label=rf"$\mu_0={mu0_fit:.3f},\;\lambda={lam_fit:.3f}$")
ax.set_xlabel(r"$N_\text{g}$", fontsize=18)
ax.set_ylabel(r"$\mu$  (convergence rate)", fontsize=18)
ax.set_yscale("log")
ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
ax.xaxis.set_minor_formatter(mticker.NullFormatter())
ax.set_xticks(gate_ctt)
ax.set_xticklabels(gate_ctt, rotation=45, ha="right")
ax.legend(ncol=1, loc="upper right", fontsize=16,
          labelspacing=0.4, handletextpad=0.4)
ax.text(0.02, 0.97, "(d)", transform=ax.transAxes,
        fontsize=18, verticalalignment="top")
ax.grid(True, alpha=0.3, which="both")
plt.tight_layout()
plt.savefig("Figs/v5_mu_vs_Ng.pdf", bbox_inches="tight", dpi=300)
plt.show()
print("Saved Figs/v5_mu_vs_Ng.pdf")
